# 05. 시간 경과와 날짜별 이력

- **theta**: 시뮬레이션 시장의 `update(다음 날)` — 시장은 그대로, 평가일만 이동. 기준 커브는 ForwardForward로 굴린다(오늘의 선도가 실현, ORE 기본 `YieldCurveRollDown`).
- **날짜별 이력**: 날짜마다 `TodaysMarket`을 만들고 **결과 숫자만 표로** 남긴다. 객체는 저장하지 않는다(원칙 10).

| 단계 | QuantTool 코드 | ORE 원본 |
|---|---|---|
| 날짜 이동 | `analytics/scenario/simmarket.py` `SimMarket.update`, `ScenarioSimMarket._update_date` | `orea/simulation/simmarket.cpp`, `QuantExt/qle/termstructures/spreadeddiscountcurve.cpp` |
| 날짜별 시장 | `data/marketdata/todaysmarket.py` | `OREData/ored/marketdata/todaysmarket.hpp` (asof마다 하나) |

> 데이터: `tests/fixtures/market_ust_sample.txt`는 ORE `market.txt` 형식(`날짜 키 값`)의 **합성 샘플**이다. 흐름을 익히는 용도이고, 숫자로 시장을 해석하지 않는다.

In [ ]:
from pathlib import Path

# 노트북이 어디서 열리든 레포 루트(pyproject.toml이 있는 곳)를 찾는다
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
SAMPLE = ROOT / "tests" / "fixtures" / "market_ust_sample.txt"  # 합성 샘플 (실제 시세 아님)

from datetime import date

import pandas as pd
import QuantLib as ql

from quanttool.analytics.app import curve_shape, curve_table
from quanttool.analytics.scenario import ScenarioSimMarket
from quanttool.data.configuration import load_config
from quanttool.data.marketdata import CSVLoader, TodaysMarket
from quanttool.data.portfolio import EngineFactory, Portfolio

cfg = load_config(ROOT / "configs")
loader = CSVLoader(SAMPLE)
market = TodaysMarket.from_config(date(2026, 10, 23), cfg, loader)
book = Portfolio.from_file(ROOT / "positions" / "ust_book.yaml")

## 1. theta: 시장 그대로, 하루 이동

In [ ]:
sim = ScenarioSimMarket(market, cfg.simmarket, use_spreaded_term_structures=True)
with sim.active():
    book.build(EngineFactory.from_config(cfg, sim))
    base = book.npvs()
sim.update(date(2026, 10, 26))  # 다음 영업일 (주말 포함 3일), 생성기 없음 → 기준 시나리오
with sim.active():
    print(
        "평가일",
        ql.Settings.instance().evaluationDate,
        " 커브 기준일",
        sim.yield_curve("UST").referenceDate(),
    )
    theta = book.npvs() - base
sim.reset()
theta.round(2)

ForwardForward에서 할인채의 하루 가치 변화는 그 하루의 선도금리만큼이다. 쿠폰채는 여기에 경과이자와 지급 쿠폰이 더해진다(NPV는 dirty 기준).

## 2. 날짜별 커브 이력
날짜마다 시장을 만들고 커브 표에서 필요한 숫자만 꺼내 표로 쌓는다. 실제 데이터가 연결되면 같은 코드로 몇 년치를 만든다.

In [ ]:
zero_hist, shape_hist = {}, {}
for d in loader.asof_dates():
    m = TodaysMarket.from_config(d, cfg, loader)
    table = curve_table(m, "UST", cfg)
    zero_hist[d] = table["zero_cc"] * 100
    shape_hist[d] = curve_shape(table)
display(pd.DataFrame(zero_hist).T.round(4))
pd.DataFrame(shape_hist).T

## 3. 평가일이 바꾸는 것과 바꾸지 않는 것
`TodaysMarket`의 커브는 asof에 고정되어 있고, 할인 엔진의 NPV도 커브 기준일로 할인하므로 전역 평가일과 무관하다. 하지만 상품의 **결제일·경과이자·clean 가격**은 평가일에서 정해진다. 쿠폰일(2027-02-15)을 지난 평가일이 남아 있으면 아래처럼 다른 숫자가 나온다. 그래서 상품 평가는 그 시장의 `active()` 안에서 한다.

In [ ]:
from quanttool.data.marketdata.market import evaluation_date

with market.active():
    book.build(EngineFactory.from_config(cfg, market))
bond10 = book.get("UST_10Y").instrument().ql_instrument()


def snapshot(label):
    return {
        "평가일": label,
        "결제일": bond10.settlementDate().ISO(),
        "경과이자": bond10.accruedAmount(),
        "clean": bond10.cleanPrice(),
        "NPV(10Y 거래)": book.get("UST_10Y").npv(),
    }


rows = []
with market.active():
    rows.append(snapshot("2026-10-23 (active)"))
with evaluation_date(date(2027, 2, 16)):  # 다른 날짜의 평가일이 남아 있는 상황
    rows.append(snapshot("2027-02-16"))
pd.DataFrame(rows).set_index("평가일").round(4)

결제일이 쿠폰일을 넘으면서 경과이자와 clean 가격은 바뀌었지만 NPV는 그대로다. 두 숫자가 서로 다른 날짜를 기준으로 섞이지 않도록, 한 시장의 결과는 그 시장의 `active()` 안에서 만든다.